In [1]:
import pandas as pd

data = pd.read_csv('Combined Data.csv')
data.head()

,Unnamed: 0,statement,status
0,0,oh my gosh,Anxiety
1,1,"trouble sleeping, confused mind, restless hear...",Anxiety
2,2,"All wrong, back off dear, forward doubt. Stay ...",Anxiety
3,3,I've shifted my focus to something else but I'...,Anxiety
4,4,"I'm restless and restless, it's been a month n...",Anxiety


In [2]:
# Logistic model - Balanced and stable

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

# Step 1: Clean data FIRST
data = data.drop(columns=['Unnamed: 0'], errors='ignore')
data.dropna(subset=['statement'], inplace=True)

# Step 2: Preprocessing function
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    words = [stemmer.stem(word) for word in words if word not in stop_words]
    return ' '.join(words)

# Apply preprocessing
data['processed_statement'] = data['statement'].apply(preprocess_text)

# Step 3: Define X and y
X_text = data['processed_statement']
y = data['status']

# Step 4: Train-test split (BEFORE vectorization)
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)

# Step 5: TF-IDF (fit ONLY on train)
tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))

X_train = tfidf_vectorizer.fit_transform(X_train_text)   # ✅ fit here
X_test = tfidf_vectorizer.transform(X_test_text)         # ✅ only transform

# Step 6: Train model (with class balancing)
log_reg_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
log_reg_model.fit(X_train, y_train)

# 🔹 Step 7: Prediction
y_pred = log_reg_model.predict(X_test)

# 🔹 Step 8: Evaluation
print("Logistic Regression Model Performance:")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\SURAJ\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Logistic Regression Model Performance:
Accuracy: 0.7601

Classification Report:

                      precision    recall  f1-score   support

             Anxiety       0.78      0.80      0.79       768
             Bipolar       0.75      0.83      0.79       556
          Depression       0.79      0.63      0.70      3081
              Normal       0.88      0.91      0.89      3269
Personality disorder       0.48      0.75      0.59       215
              Stress       0.50      0.67      0.57       517
            Suicidal       0.67      0.72      0.69      2131

            accuracy                           0.76     10537
           macro avg       0.69      0.76      0.72     10537
        weighted avg       0.77      0.76      0.76     10537



# SVM model

often best classical model

In [3]:
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score

svm_model = LinearSVC(class_weight='balanced', random_state=42)

svm_model.fit(X_train, y_train)

y_pred_svm = svm_model.predict(X_test)

print("SVM Model Performance:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_svm):.4f}")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_svm))

SVM Model Performance:
Accuracy: 0.7533

Classification Report:

                      precision    recall  f1-score   support

             Anxiety       0.75      0.78      0.76       768
             Bipolar       0.75      0.81      0.78       556
          Depression       0.75      0.65      0.70      3081
              Normal       0.87      0.92      0.90      3269
Personality disorder       0.56      0.71      0.63       215
              Stress       0.50      0.58      0.54       517
            Suicidal       0.66      0.67      0.66      2131

            accuracy                           0.75     10537
           macro avg       0.69      0.73      0.71     10537
        weighted avg       0.75      0.75      0.75     10537



# Navie Byers
Fast + works very well with TF-IDF


Fast but slightly weaker

In [4]:
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()

nb_model.fit(X_train, y_train)

y_pred_nb = nb_model.predict(X_test)

print("Naive Bayes Model Performance:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_nb):.4f}")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_nb))

Naive Bayes Model Performance:
Accuracy: 0.6908

Classification Report:

                      precision    recall  f1-score   support

             Anxiety       0.77      0.66      0.71       768
             Bipolar       0.85      0.56      0.68       556
          Depression       0.57      0.77      0.66      3081
              Normal       0.80      0.85      0.82      3269
Personality disorder       0.97      0.14      0.24       215
              Stress       0.72      0.13      0.22       517
            Suicidal       0.70      0.56      0.62      2131

            accuracy                           0.69     10537
           macro avg       0.77      0.53      0.57     10537
        weighted avg       0.71      0.69      0.68     10537



# Deep Learning LSTM

In [6]:
# Tokenization
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_words = 10000
max_len = 100

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(X_train_text)

X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)

In [7]:
# Encode labels

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

In [10]:
# Build model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Input

model = Sequential([
    Input(shape=(max_len,)),
    Embedding(input_dim=max_words, output_dim=128),
    LSTM(64, return_sequences=False),
    Dense(32, activation='relu'),
    Dense(len(set(y)), activation='softmax')
])

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 100, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │           231 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,331,719 (5.08 MB)

 Trainable params: 1,331,719 (5.08 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
# Train 
model.fit(
    X_train_pad,
    y_train_enc,
    epochs=5,
    batch_size=64,
    validation_split=0.1
)

Epoch 1/5
593/593 ━━━━━━━━━━━━━━━━━━━━ 111s 175ms/step - accuracy: 0.6442 - loss: 0.9144 - val_accuracy: 0.7288 - val_loss: 0.7029
Epoch 2/5
593/593 ━━━━━━━━━━━━━━━━━━━━ 141s 173ms/step - accuracy: 0.7664 - loss: 0.6110 - val_accuracy: 0.7469 - val_loss: 0.6643
Epoch 3/5
593/593 ━━━━━━━━━━━━━━━━━━━━ 138s 166ms/step - accuracy: 0.8107 - loss: 0.5000 - val_accuracy: 0.7554 - val_loss: 0.6339
Epoch 4/5
593/593 ━━━━━━━━━━━━━━━━━━━━ 53s 89ms/step - accuracy: 0.8413 - loss: 0.4190 - val_accuracy: 0.7526 - val_loss: 0.6484
Epoch 5/5
593/593 ━━━━━━━━━━━━━━━━━━━━ 51s 87ms/step - accuracy: 0.8698 - loss: 0.3525 - val_accuracy: 0.7635 - val_loss: 0.6852


In [12]:
# Evaluate
loss, acc = model.evaluate(X_test_pad, y_test_enc)
print(f"LSTM Accuracy: {acc:.4f}")

330/330 ━━━━━━━━━━━━━━━━━━━━ 8s 23ms/step - accuracy: 0.7497 - loss: 0.7028
LSTM Accuracy: 0.7497


In [13]:
import pickle

pickle.dump(log_reg_model, open("model.pkl", "wb"))
pickle.dump(tfidf_vectorizer, open("vectorizer.pkl", "wb"))